# Practical Exercise with MNIST Example

In [1]:
import torch # 파이토치 라이브러리
import torch.nn # 신경망 모듈

In [2]:
import sys # 파이썬 인터프리터 제어 모듈
import numpy as np # 수치 계산 모듈
import matplotlib.pyplot as plt # 시각화 모듈

from mnist_classifier.utils import load_mnist # MNIST 로드 함수
from mnist_classifier.utils import get_model # 모델 생성 함수

In [3]:
model_fn = "./model.pth" # 모델 가중치 파일 경로 설정

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu') # 장치 설정

In [5]:
def load(fn, device): # 모델 로드 함수 정의
    d = torch.load(fn, map_location=device, weights_only=False) # 파일 로드
    
    return d['model'], d['config'] # 모델 상태와 설정값 반환

In [6]:
def plot(x, y_hat): # 결과 시각화 함수 정의
    for i in range(x.size(0)): # 배치 크기만큼 반복
        img = (np.array(x[i].detach().cpu(), dtype='float')).reshape(28,28) # 이미지 재구성

        plt.imshow(img, cmap='gray') # 이미지 표시
        plt.show() # 화면 출력
        print("Predict:", float(torch.argmax(y_hat[i], dim=-1))) # 예측값 출력

In [7]:
def test(model, x, y, to_be_shown=True): # 테스트 수행 함수 정의
    model.eval() # 평가 모드 설정
    
    with torch.no_grad(): # 기울기 계산 제외
        y_hat = model(x) # 예측 수행

        correct_cnt = (y.squeeze() == torch.argmax(y_hat, dim=-1)).sum() # 정답 수 계산
        total_cnt = float(x.size(0)) # 전체 수
        
        accuracy = correct_cnt / total_cnt # 정확도 계산
        print("Accuracy: %.4f" % accuracy) # 결과 출력
        
        if to_be_shown: # 시각화 여부 확인
            plot(x, y_hat) # 시각화 호출

In [8]:
model_dict, train_config = load(model_fn, device) # 모델 및 설정 로드

print(train_config) # 설정 출력

In [9]:
# Load MNIST test set.
x, y = load_mnist(is_train=False, flatten=(train_config.model == "fc")) # 테스트 데이터 로드
x, y = x.to(device), y.to(device) # 장치 전송

print(x.shape, y.shape) # 데이터 형태 출력

input_size = int(x.shape[-1]) # 입력 크기
output_size = int(max(y)) + 1 # 출력 크기

model = get_model( # 모델 생성
    input_size,
    output_size,
    train_config,
    device,
)

model.load_state_dict(model_dict) # 가중치 로드

test(model, x, y, to_be_shown=False) # 테스트 수행

In [10]:
n_test = 20 # 테스트할 샘플 수
test(model, x[:n_test], y[:n_test], to_be_shown=True) # 일부 샘플 테스트 및 시각화